# Долгий запуск установки со включенной стабилизацией, использующей ПИД-регулятор 

In [ ]:
from nn_laser_stabilizer.utils.paths import get_data_dir

import pandas as pd

df = pd.read_csv(
    get_data_dir() / "250626 PID стабилизация.csv",
    skiprows=[1],      # строка с единицами измерения
    dtype=str
)

df = df.rename(columns={
    "Время": "time",
    "Uфот": "process_variable",
    "setpoint": "setpoint",
    "Uфазовр": "control_output",
})

df["time"] = pd.to_datetime(
    df["time"],
    format="%H:%M:%S.%f",
    errors="coerce",
)

for col in ["process_variable", "setpoint", "control_output"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# есть несколько некорректных строк
df = df.dropna(
    subset=[
        "time",
        "process_variable",
        "setpoint",
        "control_output",
    ]
).reset_index(drop=True)

df["process_variable"] /= 10
df["setpoint"] /= 10

df["time_min"] = (
    df["time"] - df["time"].iloc[0]
).dt.total_seconds() / 60

df = df[[
    "time_min",
    "process_variable",
    "setpoint",
    "control_output",
]]

In [ ]:
df.info()

In [ ]:
def hsl_to_rgb_normalized(h, s, l):
    # colorsys использует HLS (не HSL), поэтому порядок (h, l, s)
    import colorsys
    return colorsys.hls_to_rgb(h / 360, l / 100, s / 100)

BLUE_RGB = hsl_to_rgb_normalized(206, 73, 48)
GREEN_RGB = hsl_to_rgb_normalized(120, 72, 44)
RED_RGB = hsl_to_rgb_normalized(9, 98, 63)
PURPLE_RGB = hsl_to_rgb_normalized(279, 98, 76)

import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
})

In [ ]:
window = 5000

t_min = df["time_min"]
pv = df["process_variable"]
co = df["control_output"]
setpoint = df["setpoint"]

rolling_mean = pv.rolling(window, center=True).mean()
rolling_std = pv.rolling(window, center=True).std()

fig, (ax_pv, ax_co) = plt.subplots(
    2, 1,
    figsize=(14, 10),
    sharex=True,
    gridspec_kw={"height_ratios": [1, 1], "hspace": 0.08},
)

ax_pv.fill_between(
    t_min,
    rolling_mean - rolling_std,
    rolling_mean + rolling_std,
    color=BLUE_RGB,
    alpha=0.2,
    label="±std",
    rasterized=True,
)

ax_pv.plot(
    t_min,
    rolling_mean,
    color=BLUE_RGB,
    linewidth=3,
    label="Photodetector",
)

ax_pv.plot(
    t_min,
    setpoint,
    "--",
    color=RED_RGB,
    linewidth=5,
    label="Setpoint",
)

ax_pv.set_ylabel("Photodetector Signal")
ax_pv.grid(False)

ax_co.plot(
    t_min,
    co,
    color=GREEN_RGB,
    linewidth=1,
    alpha=0.7,
)

ax_co.set_ylabel("Control Signal")
ax_co.set_xlabel("Time, min")
ax_co.grid(False)

ax_pv.set_ylim(101, 151)
ax_co.set_ylim(-99, 4109)

ax_pv.text(
    -0.1, 1.05, "(a)",
    transform=ax_pv.transAxes,
    fontsize=30,
    va="center",
    ha="center",
)

ax_co.text(
    -0.1, 1.02, "(b)",
    transform=ax_co.transAxes,
    fontsize=30,
    va="center",
    ha="center",
)

handles, labels = ax_pv.get_legend_handles_labels()
ax_pv.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=2,
    frameon=False,
)

fig.align_ylabels([ax_pv, ax_co])